In [1]:
import pandas as pd

In [2]:
df= pd.read_csv(r'C:\Users\enesm\OneDrive\Masaüstü\My_Deep_Learning_Journey\FastAi\patent_data\train.csv')

In [3]:
df

,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00
...,...,...,...,...,...
36468,8e1386cbefd7f245,wood article,wooden article,B44,1.00
36469,42d9e032d1cd3242,wood article,wooden box,B44,0.50
36470,208654ccb9e14fa3,wood article,wooden handle,B44,0.50
36471,756ec035e694722b,wood article,wooden material,B44,0.75


In [4]:
df.describe(include='object')

,id,anchor,target,context
count,36473,36473,36473,36473
unique,36473,733,29340,106
top,37d61fd2272659b1,component composite coating,composition,H01
freq,1,152,24,2186


In [5]:
df['input']='TEXT1: ' + df.context + '; TEXT2: '+ df.target + '; ANC1: '+ df.anchor

In [6]:
df.input.head()

0    TEXT1: A47; TEXT2: abatement of pollution; ANC...
1    TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2    TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3    TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4    TEXT1: A47; TEXT2: forest region; ANC1: abatement
Name: input, dtype: object

## Tokenization

In [7]:
import datasets

Transformers uses a Dataset object for storing a dataset


In [8]:
from datasets import Dataset,DatasetDict

ds=Dataset.from_pandas(df)

In [9]:
ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input'],
    num_rows: 36473
})

In [10]:
model_nm='microsoft/deberta-v3-small'

In [11]:
import sentencepiece
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz=AutoTokenizer.from_pretrained(model_nm)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\convert_slow_tokenizer.py:454: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [12]:
tokz.tokenize(" G'day folks, I'm Malik from fast.ai! ")

['▁G',
 "'",
 'day',
 '▁folks',
 ',',
 '▁I',
 "'",
 'm',
 '▁Malik',
 '▁from',
 '▁fast',
 '.',
 'ai',
 '!']

In [13]:
tokz.tokenize("A platypus is an ornithorhynchus anatinus.")

['▁A',
 '▁platypus',
 '▁is',
 '▁an',
 '▁or',
 'ni',
 'tho',
 'rhynch',
 'us',
 '▁an',
 'at',
 'inus',
 '.']

In [14]:
def tok_func(x): return tokz(x['input'])

In [15]:
tokz('hello world') 

{'input_ids': [1, 12018, 447, 2], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [16]:
tokz.tokenize('hello world')

['▁hello', '▁world']

In [17]:
ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input'],
    num_rows: 36473
})

In [18]:
tok_ds=ds.map(tok_func,batched=True)

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [19]:
tok_ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 36473
})

In [20]:
tok_ds=tok_ds.rename_column('score','labels')

In [21]:
row=tok_ds[0]
row['input'],row['input_ids'],row['token_type_ids'],row['attention_mask'],row['labels']                                                               

('TEXT1: A47; TEXT2: abatement of pollution; ANC1: abatement',
 [1,
  54453,
  435,
  294,
  336,
  5753,
  346,
  54453,
  445,
  294,
  47284,
  265,
  6435,
  346,
  23702,
  435,
  294,
  47284,
  2],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 0.5)

In [22]:
tokz.vocab['47']

5753

In [23]:
dds= tok_ds.train_test_split(0.25,seed=42)
dds

DatasetDict({
    train: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27354
    })
    test: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9119
    })
})

# Validation

In [24]:
eval_df=pd.read_csv(r'C:\Users\enesm\OneDrive\Masaüstü\My_Deep_Learning_Journey\FastAi\patent_data\test.csv')
eval_df.describe(include='object')

,id,anchor,target,context
count,36,36,36,36
unique,36,34,36,29
top,4112d61851461f60,el display,inorganic photoconductor drum,G02
freq,1,2,1,3


In [25]:
eval_df['input']='TEXT1: ' + eval_df.context + '; TEXT2: '+  eval_df.target + '; ANC1: '+ eval_df.anchor

In [26]:
ds_val=Dataset.from_pandas(eval_df)

In [27]:
eval_ds=ds_val.map(tok_func,batched=True)

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [28]:
dds

DatasetDict({
    train: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27354
    })
    test: Dataset({
        features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9119
    })
})

In [29]:
eval_ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 36
})

# Training

In [30]:
from transformers import Trainer,TrainingArguments

In [31]:
bs=128
epochs=6

In [32]:
lr=8e-5

In [33]:
import torch 

In [34]:
import torch
from torch import device

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [36]:
dds['train']

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'labels', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 27354
})

In [37]:
ds_val

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'input'],
    num_rows: 36
})

In [38]:
args=TrainingArguments('outputs',learning_rate=lr,warmup_ratio=0.1,lr_scheduler_type='cosine',
fp16=True,
    evaluation_strategy='epoch',per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs,weight_decay=0.01,report_to='none')

In [39]:
model=AutoModelForSequenceClassification.from_pretrained(model_nm,num_labels=1)

Some weights of the model checkpoint at microsoft/deberta-v3-small were not used when initializing DebertaV2ForSequenceClassification: ['lm_predictions.lm_head.bias', 'lm_predictions.lm_head.dense.weight', 'lm_predictions.lm_head.dense.bias', 'mask_predictions.classifier.bias', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.LayerNorm.weight', 'mask_predictions.classifier.weight', 'mask_predictions.dense.weight', 'mask_predictions.LayerNorm.weight', 'mask_predictions.dense.bias', 'lm_predictions.lm_head.LayerNorm.bias']
- This IS expected if you are initializing DebertaV2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from 

In [40]:
import numpy as np 

In [41]:
def corr(x,y): return np.corrcoef(x,y)[0][1]

In [42]:
def corr_d(eval_pred): return {'pearson': corr(*eval_pred)}

In [43]:
trainer=Trainer(model=model,args=args,train_dataset=dds['train'],eval_dataset=dds['test'],
                tokenizer=tokz,compute_metrics=corr_d)

In [50]:
import nvidia_smi

In [51]:
nvidia_smi

<module 'nvidia_smi' from 'c:\\Users\\enesm\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\nvidia_smi.py'>

In [52]:
trainer.train()

  0%|          | 0/1284 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 0.022084003314375877, 'eval_pearson': 0.8227923186517471, 'eval_runtime': 2.1918, 'eval_samples_per_second': 4160.47, 'eval_steps_per_second': 16.425, 'epoch': 1.0}


  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 0.022165663540363312, 'eval_pearson': 0.8333290979576429, 'eval_runtime': 2.1925, 'eval_samples_per_second': 4159.136, 'eval_steps_per_second': 16.419, 'epoch': 2.0}
{'loss': 0.0149, 'learning_rate': 2.2988519847102787e-05, 'epoch': 2.34}


...

  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 0.02211221493780613, 'eval_pearson': 0.8382498447206639, 'eval_runtime': 2.2431, 'eval_samples_per_second': 4065.283, 'eval_steps_per_second': 16.049, 'epoch': 5.0}


  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 0.02192668430507183, 'eval_pearson': 0.8384093334928173, 'eval_runtime': 2.2536, 'eval_samples_per_second': 4046.4, 'eval_steps_per_second': 15.974, 'epoch': 6.0}
{'train_runtime': 220.4435, 'train_samples_per_second': 744.517, 'train_steps_per_second': 5.825, 'train_loss': 0.011419686015892622, 'epoch': 6.0}


TrainOutput(global_step=1284, training_loss=0.011419686015892622, metrics={'train_runtime': 220.4435, 'train_samples_per_second': 744.517, 'train_steps_per_second': 5.825, 'train_loss': 0.011419686015892622, 'epoch': 6.0})

In [63]:
preds=trainer.predict(eval_ds).predictions.astype(float)

  0%|          | 0/1 [00:00<?, ?it/s]

In [65]:
preds=preds.clip(0,1)

In [62]:
preds

array([[0.47558594],
       [0.5859375 ],
       [0.58398438],
       [0.31591797],
       [0.        ],
       [0.53320312],
       [0.5234375 ],
       [0.07617188],
       [0.359375  ],
       [1.        ],
       [0.28833008],
       [0.25756836],
       [0.76904297],
       [0.96484375],
       [0.77392578],
       [0.43505859],
       [0.23352051],
       [0.        ],
       [0.66015625],
       [0.38574219],
       [0.60107422],
       [0.27539062],
       [0.12115479],
       [0.24023438],
       [0.57373047],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.48950195],
       [0.29003906],
       [0.03268433],
       [0.75732422],
       [0.48217773],
       [0.41894531],
       [0.26782227]])

In [66]:
submission=datasets.Dataset.from_dict({
    'id':eval_ds['id'],
    'score':preds
})

submission.to_csv('submission.csv',index=False)

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1066